<a href="https://colab.research.google.com/github/costpetrides/FAIRMODE-WG5/blob/main/Phase2/JRC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import netCDF4 as nc
import numpy as np
import os

input_dir = "/content/O3"
output_dir = "/content/Results_O3"

variables_to_fix = [
    "SURF_ug_NO2",
    "SURF_ug_O3",              # corrected name
    "SURF_ug_PM25_rh50"
]

correct_dims = ("time", "lat", "lon")

os.makedirs(output_dir, exist_ok=True)

for file in os.listdir(input_dir):
    if file.endswith(".nc"):

        input_path = os.path.join(input_dir, file)
        output_path = os.path.join(output_dir, file)

        print(f"\nProcessing: {file}")

        with nc.Dataset(input_path, "r") as src:
            lon = src.variables["lon"][:]
            lat = src.variables["lat"][:]
            time = src.variables["time"][:]

            global_attrs = {attr: src.getncattr(attr) for attr in src.ncattrs()}

            with nc.Dataset(output_path, "w", format="NETCDF4") as dst:

                # === FIXED DIMENSIONS ===
                dst.createDimension("lon", len(lon))
                dst.createDimension("lat", len(lat))
                dst.createDimension("time", len(time))   # NOT unlimited

                # === COORDINATES (correct dtype) ===
                lon_var = dst.createVariable("lon", "f8", ("lon",))
                lat_var = dst.createVariable("lat", "f8", ("lat",))
                time_var = dst.createVariable("time", "f8", ("time",))

                lon_var[:] = lon
                lat_var[:] = lat
                time_var[:] = time

                # Copy coordinate attributes
                for coord in ["lon", "lat", "time"]:
                    for attr in src.variables[coord].ncattrs():
                        getattr(dst.variables[coord], "setncattr")(
                            attr, src.variables[coord].getncattr(attr)
                        )

                # === Global attributes ===
                for attr, value in global_attrs.items():
                    dst.setncattr(attr, value)

                # === Variables ===
                for var_name in variables_to_fix:

                    if var_name not in src.variables:
                        print(f"  ⚠ {var_name} not found.")
                        continue

                    var = src.variables[var_name]
                    dims = var.dimensions
                    data = var[:]
                    var_attrs = {a: var.getncattr(a) for a in var.ncattrs()}

                    # === Reshape if needed ===
                    if dims != correct_dims:

                        print(f"   Reshaping {var_name}: {dims} → {correct_dims}")

                        if dims == ("lon", "lat", "time"):
                            data = np.transpose(data, (2, 1, 0))
                        elif dims == ("lat", "lon", "time"):
                            data = np.transpose(data, (2, 0, 1))
                        else:
                            print(f"   ⚠ Unknown dimension order {dims}")
                            continue

                    # === Write variable ===
                    fill_value = var_attrs.pop("_FillValue", -9999.0)

                    var_out = dst.createVariable(
                        var_name,
                        "f4",
                        correct_dims,
                        fill_value=fill_value
                    )

                    for attr, value in var_attrs.items():
                        var_out.setncattr(attr, value)

                    var_out[:] = data

        print(f"  ✔ Saved to {output_path}")

print("\n✔ All files processed correctly.")


Processing: Scen_2022_NKUA_O3_CSA.Cell.Add.ADI_CORR_YEARLY.nc
  ⚠ SURF_ug_NO2 not found.
   Reshaping SURF_ug_O3: ('lon', 'lat', 'time') → ('time', 'lat', 'lon')
  ⚠ SURF_ug_PM25_rh50 not found.
  ✔ Saved to /content/Results_O3/Scen_2022_NKUA_O3_CSA.Cell.Add.ADI_CORR_YEARLY.nc

Processing: Scen_2024_NKUA_O3_CSA.Cell.Add.ADI_CORR_YEARLY.nc
  ⚠ SURF_ug_NO2 not found.
   Reshaping SURF_ug_O3: ('lon', 'lat', 'time') → ('time', 'lat', 'lon')
  ⚠ SURF_ug_PM25_rh50 not found.
  ✔ Saved to /content/Results_O3/Scen_2024_NKUA_O3_CSA.Cell.Add.ADI_CORR_YEARLY.nc

Processing: BaseCase_NKUA_O3_CSA.Cell.Add.ADI_CORR_YEARLY.nc
  ⚠ SURF_ug_NO2 not found.
   Reshaping SURF_ug_O3: ('lon', 'lat', 'time') → ('time', 'lat', 'lon')
  ⚠ SURF_ug_PM25_rh50 not found.
  ✔ Saved to /content/Results_O3/BaseCase_NKUA_O3_CSA.Cell.Add.ADI_CORR_YEARLY.nc

Processing: Scen_2023_NKUA_O3_CSA.Cell.Add.ADI_CORR_YEARLY.nc
  ⚠ SURF_ug_NO2 not found.
   Reshaping SURF_ug_O3: ('lon', 'lat', 'time') → ('time', 'lat', 'lon')
  